<a href="https://colab.research.google.com/github/gorkyKirill/My_project/blob/Hack'24/HackatonMIR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install onnxruntime

In [ ]:
import onnxruntime as ort
import numpy as np
from PIL import Image
from torchvision import transforms

# Загрузка модели ONNX
onnx_model_path = "/content/drive/MyDrive/Hackaton/webface_r50_pfc.onnx"
ort_session = ort.InferenceSession(onnx_model_path)

# Преобразование изображения к формату, необходимому для модели ResNet50
preprocess = transforms.Compose([
    transforms.Resize(128),  # Измените размер на что-то большее, чем 112, чтобы потом сделать центрирование
    transforms.CenterCrop(112),  # Изменение размера на 112x112
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def to_numpy(tensor):
    """Перевод тензора в формат numpy для ONNX."""
    return tensor.detach().cpu().numpy() if tensor.requires_grad else tensor.cpu().numpy()

def get_embedding(image_path):
    # Загрузка и предобработка изображения
    img = Image.open(image_path)
    img_tensor = preprocess(img).unsqueeze(0)  # Добавляем batch dimension

    # Переводим тензор в формат numpy
    input_image = to_numpy(img_tensor)

    # Запускаем модель для извлечения эмбеддингов
    ort_inputs = {ort_session.get_inputs()[0].name: input_image}
    ort_outs = ort_session.run(None, ort_inputs)

    # Получаем результат - эмбеддинги
    embedding = ort_outs[0]

    return embedding

# Пример использования: путь к изображению
image_path = "path_to_your_image.jpg"
embedding = get_embedding(image_path)
print("Embedding shape:", embedding.shape)


In [9]:
import onnxruntime as ort

# Загрузка модели ONNX
onnx_model_path = "/content/drive/MyDrive/Hackaton/webface_r50_pfc.onnx"
ort_session = ort.InferenceSession(onnx_model_path)

# Получение информации о входных данных модели
input_name = ort_session.get_inputs()[0].name
input_shape = ort_session.get_inputs()[0].shape
input_type = ort_session.get_inputs()[0].type

print(f"Input name: {input_name}")
print(f"Input shape: {input_shape}")
print(f"Input type: {input_type}")

# Получение информации о выходных данных модели
output_name = ort_session.get_outputs()[0].name
output_shape = ort_session.get_outputs()[0].shape
output_type = ort_session.get_outputs()[0].type

print(f"Output name: {output_name}")
print(f"Output shape: {output_shape}")
print(f"Output type: {output_type}")

Input name: input.1
Input shape: ['None', 3, 112, 112]
Input type: tensor(float)
Output name: 683
Output shape: [1, 512]
Output type: tensor(float)
